# Abfragen nach Kategorien und Artikeln

## 1. Zeige alphabetisch geordnet alle Artikel an, die keine Auslaufartikel sind und deren Lagerbestand unter dem Mindestbestand liegt.

![Ka-0010](Screenshots/Ka-0010.png)

In [ ]:
--
SELECT artikel_id as 'ID', bezeichnung as 'Artikel', mindestbestand, lagerbestand
FROM artikel
WHERE auslaufartikel = '0'
AND lagerbestand < mindestbestand
ORDER BY bezeichnung;

(17 rows affected)

ID | Artikel                   | mindestbestand | lagerbestand
---+---------------------------+----------------+-------------
3  | Aniseed Syrup             | 25             | 13          
2  | Chang                     | 25             | 17          
48 | Chocolade                 | 25             | 15          
56 | Gnocchi di nonna Alice    | 30             | 21          
37 | Gravad lax                | 25             | 11          
43 | Ipoh Coffee               | 25             | 17          
74 | Longlife Tofu             | 5              | 4           
66 | Louisiana Hot Spiced Okra | 20             | 4           
32 | Mascarpone Fabioli        | 25             | 9           
49 | Maxilaku                  | 15             | 10          
30 | Nord-Ost Matjeshering     | 15             | 10          
70 | Outback Lager             | 30             | 15          
11 | Queso Cabrales            | 30             | 22          
45 | Røgede sild               | 15

## 2. Ermittle zu der vorherigen Abfrage die notwendige Nachbestellung, damit der neue Lagerbestand das Doppelte des Mindestbestands ist.

![Ka-0020](Screenshots/Ka-0020.png)

In [39]:
--
SELECT
artikel_id AS 'ID',
bezeichnung AS 'Artikel',
mindestbestand,
lagerbestand,
(2 * mindestbestand) - lagerbestand as 'Nachbestellung'
FROM artikel
WHERE auslaufartikel = '0'
AND lagerbestand < mindestbestand
ORDER BY bezeichnung;

(17 rows affected)

ID | Artikel                   | mindestbestand | lagerbestand | Nachbestellung
---+---------------------------+----------------+--------------+---------------
3  | Aniseed Syrup             | 25             | 13           | 37            
2  | Chang                     | 25             | 17           | 33            
48 | Chocolade                 | 25             | 15           | 35            
56 | Gnocchi di nonna Alice    | 30             | 21           | 39            
37 | Gravad lax                | 25             | 11           | 39            
43 | Ipoh Coffee               | 25             | 17           | 33            
74 | Longlife Tofu             | 5              | 4            | 6             
66 | Louisiana Hot Spiced Okra | 20             | 4            | 36            
32 | Mascarpone Fabioli        | 25             | 9            | 41            
49 | Maxilaku                  | 15             | 10           | 20            
30 | Nord-Ost Matjes

## 3. Finde den Artikel, der noch niemals bestellt wurde.

Finde mehrere Lösungswege zu dieser Aufgabe.

![Ka-0030](Screenshots/Ka-0030.png)

In [ ]:
-- Lösung mit eigenständiger Unterabfrage


In [ ]:
-- Lösung mit korrelierter Unterabfrage


In [44]:
-- Lösung mit einem JOIN
SELECT
  a.artikel_id AS [ID],
  a.bezeichnung AS [Artikel]
FROM artikel AS a
LEFT JOIN bestelldetail AS bd ON bd.artikel_id = a.artikel_id
WHERE bd.artikel_id IS NULL
ORDER BY a.artikel_id

(1 row affected)

ID | Artikel      
---+--------------
78 | Beluga Kaviar
(1 row)

Total execution time: 00:00:00.004

## 4. Zeige die Namen der Produktkategorien und die Namen der zugehörigen Artikel an.

Falls Kategorien ohne zugeordnete Artikel existieren, sollen diese auch angezeigt werden.

Sortiere nach Kategorie und Artikel aufsteigend. (78 Zeilen)

![Ka-0040](Screenshots/Ka-0040.png)

In [ ]:
-- Lösung mit JOIN
SELECT
  k.bezeichnung AS [Kategorie],
  a.bezeichnung AS [Artikel]
FROM kategorie AS k
LEFT JOIN artikel AS a ON a.kategorie_id = k.kategorie_id
ORDER BY k.bezeichnung, a.bezeichnung

In [ ]:
-- Lösung mit Unterabfrage


## 5. Zeige die Namen der Produktkategorien und die Anzahl zugeordneter Artikel an.

Filtere Kategorien mit über 10 Artikeln.

![Ka-0050](Screenshots/Ka-0050.png)

In [62]:
-- Lösung mit JOIN
SELECT
  k.bezeichnung AS [Kategorie],
  COUNT(a.artikel_id) AS [Anzahl Artikel in Kategorie]
FROM kategorie AS k
LEFT JOIN artikel AS a ON a.kategorie_id = k.kategorie_id
GROUP BY k.bezeichnung
HAVING COUNT(a.artikel_id) > 10

(4 rows affected)

Kategorie     | Anzahl Artikel in Kategorie
--------------+----------------------------
Getränke      | 12                         
Gewürze       | 12                         
Süßwaren      | 13                         
Meeresfrüchte | 13                         
(4 rows)

Total execution time: 00:00:00.018

In [ ]:
-- Lösung mit Unterabfrage


In [ ]:
-- Weitere Lösungsidee


## 6. Zeige die Namen aller Produktkategorien und den Umsatz dieser Kategorien im Jahr 2015.

![Ka-0060](Screenshots/Ka-0060.png)

In [1]:
-- Komplexer JOIN
SELECT
  k.bezeichnung AS [Kategorie],
  SUM(CASE WHEN b.bestellung_id IS NOT NULL THEN bd.verkaufspreis * bd.anzahl ELSE 0 END) AS [Umsatz 2015]
FROM kategorie AS k
LEFT JOIN artikel AS a ON k.kategorie_id = a.kategorie_id
LEFT JOIN bestelldetail AS bd ON a.artikel_id = bd.artikel_id
LEFT JOIN bestellung AS b ON bd.bestellung_id = b.bestellung_id AND YEAR(b.bestelldatum) = 2015
GROUP BY k.bezeichnung

(8 rows affected)

Kategorie        | Umsatz 2015
-----------------+------------
Getränke         | 189,00     
Milchprodukte    | 0,00       
Getreideprodukte | 180,00     
Meeresfrüchte    | 0,00       
Süßwaren         | 0,00       
Fleischprodukte  | 0,00       
Gewürze          | 2242,00    
Naturprodukte    | 230,00     
(8 rows)

Total execution time: 00:00:00.148

## 7. Zeige die Namen der Produktkategorien und der Artikel, die insgesamt bestellte Menge, die durchschnittliche Bestellmenge und den erzielten Gesamtumsatz an. (78 Zeilen)

![Ka-0070](Screenshots/Ka-0070.png)

In [40]:
-- Lösung mit JOIN
SELECT
  k.bezeichnung AS Kategorie,
  a.bezeichnung AS Artikel,
  COALESCE(SUM(bd.anzahl), 0) AS [Bestellte Menge gesamt],
  FORMAT(CAST(COALESCE(SUM(bd.anzahl), 0) AS DECIMAL(10,2)) / NULLIF(COUNT(bd.anzahl), 0),'N1', 'de-DE') AS [Durchschnittliche Bestellmenge],
  FORMAT(COALESCE(SUM(bd.verkaufspreis * bd.anzahl), 0), 'C', 'de-DE') AS [Gesamtumsatz]
FROM kategorie AS k
JOIN artikel AS a ON k.kategorie_id = a.kategorie_id
LEFT JOIN bestelldetail AS bd ON a.artikel_id = bd.artikel_id
LEFT JOIN bestellung AS b ON bd.bestellung_id = b.bestellung_id
GROUP BY k.bezeichnung, a.bezeichnung
ORDER BY k.bezeichnung, a.bezeichnung

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(78 rows affected)

Kategorie        | Artikel                          | Bestellte Menge gesamt | Durchschnittliche Bestellmenge | Gesamtumsatz
-----------------+----------------------------------+------------------------+--------------------------------+-------------
Fleischprodukte  | Alice Mutton                     | 978                    | 26,4                           | 35.482,20 € 
Fleischprodukte  | Mishi Kobe Niku                  | 95                     | 19,0                           | 8.827,00 €  
Fleischprodukte  | Pâté chinois                     | 903                    | 27,4                           | 19.512,00 € 
Fleischprodukte  | Perth Pasties                    | 722                    | 24,1                           | 21.510,20 € 
Fleischprodukte  | Thüringer Rostbratwurst          | 746                    | 23,3                           | 87.736,40 € 
Fleischprodukte  |

## 8. Zeige für jede Produktkategorie die beiden Produkte an, die zuletzt bestellt wurden.

![Ka-0080](Screenshots/Ka-0080.png)

In [46]:
--
WITH Ranked AS (
    SELECT
        k.bezeichnung AS Kategorie,
        a.bezeichnung AS Artikel,
        b.bestelldatum AS Datum,
        ROW_NUMBER() OVER (PARTITION BY k.bezeichnung ORDER BY b.bestelldatum DESC) AS Rang
    FROM kategorie AS k
    JOIN artikel AS a ON k.kategorie_id = a.kategorie_id
    LEFT JOIN bestelldetail AS bd ON a.artikel_id = bd.artikel_id
    LEFT JOIN bestellung AS b ON bd.bestellung_id = b.bestellung_id
)
SELECT Kategorie, Artikel, Datum
FROM Ranked
WHERE Rang <= 2
ORDER BY Kategorie, Datum DESC;

(16 rows affected)

Kategorie        | Artikel                         | Datum     
-----------------+---------------------------------+-----------
Fleischprodukte  | Pâté chinois                    | 2014-05-06
Fleischprodukte  | Alice Mutton                    | 2014-05-02
Getränke         | Steeleye Stout                  | 2015-01-21
Getränke         | Chang                           | 2014-06-27
Getreideprodukte | Tunnbröd                        | 2015-01-21
Getreideprodukte | Tunnbröd                        | 2015-01-15
Gewürze          | Chef Anton's Gumbo Mix          | 2015-01-15
Gewürze          | Original Frankfurter grüne Soße | 2014-05-06
Meeresfrüchte    | Røgede sild                     | 2014-12-17
Meeresfrüchte    | Ikura                           | 2014-05-06
Milchprodukte    | Camembert Pierrot               | 2014-06-24
Milchprodukte    | Mascarpone Fabioli              | 2014-05-06
Naturprodukte    | Tofu                            | 2015-01-15
Naturprodukte    | T

## 9. Zeige für jede Produktkategorie die drei Produkte mit dem höchsten erzielten Umsatz an.

Formatiere den Umsatz als Currency. **(24 Zeilen)**

![Ka-0090](.\Screenshots\Ka-0090.png)

In [54]:
--
SELECT
    Kategorie,
    Artikel,
    FORMAT(Umsatz, 'C', 'de-DE') AS [Umsatz]
FROM (
    SELECT
        k.bezeichnung AS Kategorie,
        a.bezeichnung AS Artikel,
        COALESCE(SUM(bd.verkaufspreis * bd.anzahl), 0) AS Umsatz,
        ROW_NUMBER() OVER (
            PARTITION BY k.bezeichnung
            ORDER BY COALESCE(SUM(bd.verkaufspreis * bd.anzahl), 0) DESC
        ) AS Rang
    FROM kategorie AS k
    JOIN artikel AS a ON k.kategorie_id = a.kategorie_id
    LEFT JOIN bestelldetail AS bd ON a.artikel_id = bd.artikel_id
    GROUP BY k.bezeichnung, a.bezeichnung
) AS Ranked
WHERE Rang <= 3
ORDER BY Kategorie, Umsatz DESC;

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(24 rows affected)

Kategorie        | Artikel                          | Umsatz      
-----------------+----------------------------------+-------------
Fleischprodukte  | Thüringer Rostbratwurst          | 87.736,40 € 
Fleischprodukte  | Alice Mutton                     | 35.482,20 € 
Fleischprodukte  | Perth Pasties                    | 21.510,20 € 
Getränke         | Ipoh Coffee                      | 25.079,20 € 
Getränke         | Chang                            | 18.787,20 € 
Getränke         | Côte de Blaye                    | 149.984,20 €
Getreideprodukte | Singaporean Hokkien Fried Mee    | 9.332,40 €  
Getreideprodukte | Gnocchi di nonna Alice           | 45.121,20 € 
Getreideprodukte | Wimmers gute Semmelknödel        | 23.009,00 € 
Gewürze          | Vegie-spread                     | 17.696,30 € 
Gewürze          | Sirop d'érable                   | 16.438,80 € 
Gewürze          | Lou

## 10. Pivotiere den Gesamtumsatz der Produktkategorien nach Kategorie in Zeile und Jahr in Spalte.

Beispiel Tabellenausdruck:

![Ka-0100](Screenshots/Ka-0100.png)

In [63]:
--
WITH Umsatzdaten AS
(
  SELECT
    k.bezeichnung AS [Kategorie],
    YEAR(b.bestelldatum) AS [Jahr],
    CAST(bd.verkaufspreis * bd.anzahl AS money) AS Umsatz
  FROM kategorie AS k
  JOIN artikel AS a ON k.kategorie_id = a.kategorie_id
  JOIN bestelldetail AS bd ON a.artikel_id = bd.artikel_id
  JOIN bestellung AS b ON bd.bestellung_id = b.bestellung_id
)
SELECT
  [Kategorie],
  COALESCE([2012],0) AS [2012],
  COALESCE([2013],0) AS [2013],
  COALESCE([2014],0) AS [2014],
  COALESCE([2015],0) AS [2015]
FROM Umsatzdaten
PIVOT
(
  SUM(Umsatz) FOR [Jahr] IN ([2012], [2013], [2014], [2015])
) AS pt
ORDER BY [Kategorie]

(8 rows affected)

Kategorie        | 2012     | 2013      | 2014      | 2015   
-----------------+----------+-----------+-----------+--------
Fleischprodukte  | 30292,20 | 87621,03  | 60275,57  | 0,00   
Getränke         | 53879,20 | 110424,00 | 122451,75 | 189,00 
Getreideprodukte | 9817,60  | 60486,95  | 30422,25  | 180,00 
Gewürze          | 19458,30 | 59679,00  | 34583,45  | 2242,00
Meeresfrüchte    | 21589,60 | 71320,65  | 48902,84  | 0,00   
Milchprodukte    | 44563,30 | 123910,80 | 82939,90  | 0,00   
Naturprodukte    | 15134,20 | 57718,55  | 32645,85  | 230,00 
Süßwaren         | 31511,60 | 87227,77  | 58359,73  | 0,00   
(8 rows)

Total execution time: 00:00:00.050